# Medical-Domain LLM Fine-Tuning with QLoRA

Fine-tune **Mistral-7B-Instruct** (swappable) on **MedQA (USMLE)** with **QLoRA**, then benchmark base vs. fine-tuned accuracy.

**Runtime → Change runtime type → GPU (T4 is enough).**

## 1. Setup

In [ ]:
# Confirm we have a GPU
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
# Clone the project (skip if you uploaded it manually)
REPO_URL = 'https://github.com/TheRedCan/medical-qlora-finetune.git'
import os
if not os.path.exists('medical-qlora-finetune'):
    !git clone $REPO_URL
%cd medical-qlora-finetune

In [ ]:
!pip install -q -r requirements.txt

## 2. Hugging Face login
Mistral is **gated** — accept the license at [huggingface.co/mistralai/Mistral-7B-Instruct-v0.3](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3) and paste a token below. To skip gating, set `BASE_MODEL` to an ungated model in the next cell and skip this one.

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted

## 3. Configuration
Override anything here before importing the package.

In [ ]:
import os
# --- pick your base model ---
os.environ['BASE_MODEL'] = 'mistralai/Mistral-7B-Instruct-v0.3'
# Ungated alternative (no HF gating, runs on 8GB):
# os.environ['BASE_MODEL'] = 'Qwen/Qwen2.5-3B-Instruct'

# --- keep the run short for a free T4; raise for better results ---
os.environ['MAX_TRAIN_SAMPLES'] = '4000'
os.environ['MAX_EVAL_SAMPLES']  = '300'
os.environ['NUM_TRAIN_EPOCHS']  = '1'

from src.config import Config
cfg = Config()
cfg.as_dict()

## 4. Baseline: how good is the model *before* fine-tuning?
We evaluate the raw 4-bit base model first so we have an honest before/after comparison.

In [ ]:
from src import evaluate
from src.train import load_base_model, load_tokenizer, _subset
from src.data import load_medqa

tokenizer = load_tokenizer(cfg)
test_ds = _subset(load_medqa(cfg)['test'], cfg.max_eval_samples)

base_model = load_base_model(cfg, for_training=False); base_model.eval()
base_results = evaluate.evaluate_model(base_model, tokenizer, test_ds, cfg)
print('Base accuracy:', round(base_results['accuracy'], 3),
      '| parse rate:', round(base_results['parse_rate'], 3))

# free VRAM before training
import gc; del base_model; gc.collect(); torch.cuda.empty_cache()

## 5. Fine-tune with QLoRA
Trains LoRA adapters with prompt-masked loss. ~20–40 min on a T4 at these settings.

In [ ]:
from src.train import train
adapter_dir = train(cfg)
print('Adapter saved to', adapter_dir)

## 6. Evaluate the fine-tuned model

In [ ]:
ft_model = evaluate.load_adapter_model(adapter_dir, cfg)
ft_results = evaluate.evaluate_model(ft_model, tokenizer, test_ds, cfg)

print('Base       accuracy:', round(base_results['accuracy'], 3))
print('Fine-tuned accuracy:', round(ft_results['accuracy'], 3))
delta = ft_results['accuracy'] - base_results['accuracy']
print(f'Improvement: {delta*100:+.1f} percentage points')

## 7. Try it — qualitative demo

In [ ]:
from src.inference import answer_question

q = ('A 25-year-old woman presents with fatigue, weight gain, cold intolerance, and constipation. Lab shows elevated TSH and low free T4. What is the most likely diagnosis?')
opts = {'A': 'Graves disease', 'B': 'Hashimoto thyroiditis',
        'C': 'Pheochromocytoma', 'D': 'Cushing syndrome'}

print(answer_question(ft_model, tokenizer, q, opts, cfg))

## 8. (Optional) Push the adapter to the Hub
```python
ft_model.push_to_hub('your-username/mistral-7b-medqa-qlora')
tokenizer.push_to_hub('your-username/mistral-7b-medqa-qlora')
```